# Yuda 9 — Hierarchical + Compound Learning
Two-stage binary classifier + Progressive resolution training

In [6]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import gc
import copy
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader

import open_clip
import warnings
warnings.filterwarnings('ignore')

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.data.dataset import get_dataloaders, get_transforms

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

Device: cuda
open_clip: 3.3.0


In [7]:
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

ALL_RESULTS = []

CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim
print(f"CLIP dim: {clip_dim}")

prompts_improved = [
    "a photo of recyclable items like paper, plastic, glass, and metal",
    "a photo of electronic waste like computers, phones, cables, and batteries",
    "a photo of organic waste like food scraps, leaves, and plants",
]

class CLIPWithTextFeatures(nn.Module):
    def __init__(self, clip_model, clip_visual, prompts, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        text_dim = len(prompts)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + text_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


def get_probs(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Inference", leave=False):
            x = x.to(DEVICE)
            all_probs.append(torch.softmax(model(x), dim=1).cpu().numpy())
    return np.concatenate(all_probs)


def train_head_only(model, train_loader, val_loader, name, epochs=10):
    model = model.to(DEVICE)
    model.freeze_encoder()
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"  E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")
    model.load_state_dict(best_state)
    result = {'name': name, 'best_val_f1': best_f1}
    torch.save(best_state, RESULTS / f'{name}.pt')
    json.dump(result, open(RESULTS / f'{name}.json', 'w'))
    return model, best_f1


def record_result(name, f1, method):
    ALL_RESULTS.append({'name': name, 'best_val_f1': f1, 'method': method})
    print(f"  ✅ {name}: {f1:.4f}")

CLIP dim: 512


---
## Base Dataloaders

In [8]:
train_loader, val_loader, val_ds = get_dataloaders(batch_size=32, num_workers=4)
df_train = train_loader.dataset.df
df_val = val_loader.dataset.df
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

clip_val_ds = SingleTransformDataset(df_val, clip_transform)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

clip_train_ds = SingleTransformDataset(df_train, clip_transform)
clip_train_loader = DataLoader(clip_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)

val_labels = np.array([_CLASS_TO_IDX[r['label']] for _, r in df_val.iterrows()])
print(f"Val labels: {len(val_labels)}")

Train: 21221 | Val: 5306
Val labels: 5306


---
## Experiment A: Two-Stage Hierarchical

Stage 1: Electronic(1) vs Not-Electronic(0+2)
Stage 2: Recyclable(0) vs Organic(2) — hanya untuk non-Electronic

In [9]:
print("=" * 60)
print("Experiment A: Two-Stage Hierarchical")
print("=" * 60)

class BinaryDataset(Dataset):
    def __init__(self, df, transform, label_map):
        self.df = df
        self.transform = transform
        self.label_map = label_map
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        orig = _CLASS_TO_IDX[row['label']]
        return self.transform(img), self.label_map[orig]

# Stage 1: Electronic=1 vs Not-Electronic=0
print("\n--- Stage 1: Electronic vs Not-Electronic ---")
label_map_s1 = {0: 0, 1: 1, 2: 0}

train_s1 = DataLoader(
    BinaryDataset(df_train, clip_transform, label_map_s1),
    batch_size=32, shuffle=True, num_workers=4, pin_memory=True
)
val_s1 = DataLoader(
    BinaryDataset(df_val, clip_transform, label_map_s1),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True
)

# Train with 2 prompts (Electronic vs Non-Electronic)
prompts_s1 = [
    "a photo of non-electronic waste like paper, plastic, food scraps",
    "a photo of electronic waste like computers, phones, cables",
]
model_s1, f1_s1 = train_head_only(
    CLIPWithTextFeatures(clip_model, clip_visual, prompts_s1, num_classes=2),
    train_s1, val_s1, 'yuda9_stage1', epochs=10
)
record_result('Hierarchical_Stage1', f1_s1, 'Hierarchical')

# Val probs for Stage 1
probs_s1 = get_probs(model_s1, val_s1)
preds_s1 = probs_s1.argmax(axis=1)  # 0=non-electronic, 1=electronic
print(f"  Predicted Electronic: {preds_s1.sum()} / {val_labels.shape[0]}")

Experiment A: Two-Stage Hierarchical

--- Stage 1: Electronic vs Not-Electronic ---


  E1: val_f1=0.9941 (best=0.9941)


  E2: val_f1=0.9948 (best=0.9948)


  E3: val_f1=0.9944 (best=0.9948)


  E4: val_f1=0.9944 (best=0.9948)


...

  E7: val_f1=0.9959 (best=0.9963)


  E8: val_f1=0.9959 (best=0.9963)


  E9: val_f1=0.9967 (best=0.9967)


  E10: val_f1=0.9955 (best=0.9967)
  ✅ Hierarchical_Stage1: 0.9967


  Predicted Electronic: 786 / 5306


In [10]:
# Stage 2: Recyclable(0) vs Organic(2) — only for non-Electronic predictions
print("\n--- Stage 2: Recyclable vs Organic ---")

# Filter to only Recyclable and Organic samples for training
df_train_bin = df_train[df_train['label'].isin(['0_Recyclable', '2_Organic'])].reset_index(drop=True)
df_val_bin = df_val[df_val['label'].isin(['0_Recyclable', '2_Organic'])].reset_index(drop=True)
print(f"  Train (R+O): {len(df_train_bin)} | Val (R+O): {len(df_val_bin)}")

label_map_s2 = {0: 0, 2: 1}  # Recyclable=0, Organic=1

train_s2 = DataLoader(
    BinaryDataset(df_train_bin, clip_transform, label_map_s2),
    batch_size=32, shuffle=True, num_workers=4, pin_memory=True
)
val_s2 = DataLoader(
    BinaryDataset(df_val_bin, clip_transform, label_map_s2),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True
)

prompts_s2 = [
    "a photo of recyclable items like paper, plastic, glass, and metal",
    "a photo of organic waste like food scraps, leaves, and plants",
]
model_s2, f1_s2 = train_head_only(
    CLIPWithTextFeatures(clip_model, clip_visual, prompts_s2, num_classes=2),
    train_s2, val_s2, 'yuda9_stage2', epochs=10
)
record_result('Hierarchical_Stage2', f1_s2, 'Hierarchical')

probs_s2 = get_probs(model_s2, val_s2)


--- Stage 2: Recyclable vs Organic ---
  Train (R+O): 18052 | Val (R+O): 4514


  E1: val_f1=0.9699 (best=0.9699)


  E2: val_f1=0.9742 (best=0.9742)


  E3: val_f1=0.9737 (best=0.9742)


  E4: val_f1=0.9782 (best=0.9782)


...

  E6: val_f1=0.9787 (best=0.9787)


  E7: val_f1=0.9789 (best=0.9789)


  E8: val_f1=0.9791 (best=0.9791)


  E9: val_f1=0.9791 (best=0.9791)


  E10: val_f1=0.9791 (best=0.9791)
  ✅ Hierarchical_Stage2: 0.9791


In [11]:
# Evaluate full pipeline: Stage 1 → Stage 2
print("--- Hierarchical Pipeline Evaluation ---")

model_s1.eval()
model_s2.eval()

final_preds = []
final_probs = []
with torch.no_grad():
    for x, _ in tqdm(clip_val_loader, desc="Hierarchical inference", leave=False):
        x = x.to(DEVICE)
        
        # Stage 1: Electronic vs Not
        logits_s1 = model_s1(x)
        probs_s1 = torch.softmax(logits_s1, dim=1)
        is_electronic = probs_s1[:, 1]  # confidence for Electronic
        
        # Stage 2: Recyclable vs Organic
        logits_s2 = model_s2(x)
        probs_s2 = torch.softmax(logits_s2, dim=1)
        
        for i in range(x.size(0)):
            if is_electronic[i].item() > 0.5:
                final_preds.append(1)
                final_probs.append([0.0, 1.0, 0.0])
            else:
                # Map: Recyclable(0)=index0, Organic(1)=index2
                if probs_s2[i, 0] > probs_s2[i, 1]:
                    final_preds.append(0)
                    final_probs.append([probs_s2[i, 0].item(), 0.0, probs_s2[i, 1].item()])
                else:
                    final_preds.append(2)
                    final_probs.append([0.0, 0.0, probs_s2[i, 1].item()])

final_preds = np.array(final_preds)
final_probs = np.array(final_probs)
f1_hier = f1_score(val_labels, final_preds, average='macro')
f1_hier_per = f1_score(val_labels, final_preds, average=None)
cm_hier = confusion_matrix(val_labels, final_preds)

record_result('Hierarchical_Pipeline', f1_hier, 'Hierarchical')
print(f"\n  Hierarchical F1: {f1_hier:.4f}")
print(f"  Per-class: [{f1_hier_per[0]:.4f}, {f1_hier_per[1]:.4f}, {f1_hier_per[2]:.4f}]")
print(f"\n  Confusion Matrix:")
print(cm_hier)
print(f"\n  Normalized:")
print((cm_hier.astype(float) / cm_hier.sum(axis=1, keepdims=True)).round(4))

del model_s1, model_s2
torch.cuda.empty_cache()
gc.collect()

--- Hierarchical Pipeline Evaluation ---


  ✅ Hierarchical_Pipeline: 0.9826

  Hierarchical F1: 0.9826
  Per-class: [0.9748, 0.9924, 0.9807]

  Confusion Matrix:
[[1956    2   42]
 [   6  783    3]
 [  51    1 2462]]

  Normalized:
[[9.780e-01 1.000e-03 2.100e-02]
 [7.600e-03 9.886e-01 3.800e-03]
 [2.030e-02 4.000e-04 9.793e-01]]


53

---
## Experiment B: Compound Learning (Progressive Resolution)
Train ConvNeXt V2 at 128px → 224px → 300px

In [12]:
print("=" * 60)
print("Experiment B: Compound (Progressive Resolution)")
print("=" * 60)

resolutions = [128, 224, 300]
model_compound = None

for i, res in enumerate(resolutions):
    print(f"\n--- Stage {i+1}: {res}x{res} ---")
    
    tfm_train = get_transforms(augment=True, img_size=res)
    tfm_val = get_transforms(augment=False, img_size=res)
    
    if model_compound is None:
        model_compound = TrashClassifier('convnextv2_tiny', num_classes=3)
    
    train_loader_res = DataLoader(
        SingleTransformDataset(df_train, tfm_train),
        batch_size=64 if res <= 224 else 32, shuffle=True, num_workers=4, pin_memory=True
    )
    val_loader_res = DataLoader(
        SingleTransformDataset(df_val, tfm_val),
        batch_size=64 if res <= 224 else 32, shuffle=False, num_workers=4, pin_memory=True
    )
    
    model_compound, f1_compound = train_head_only(
        model_compound, train_loader_res, val_loader_res,
        f'yuda9_compound_{res}', epochs=10
    )
    record_result(f'Compound_{res}px', f1_compound, 'Compound')

# Final evaluation at 300px
probs_compound = get_probs(model_compound, val_loader_res)
preds_compound = probs_compound.argmax(axis=1)
f1_compound_final = f1_score(val_labels, preds_compound, average='macro')
f1_compound_per = f1_score(val_labels, preds_compound, average=None)
cm_compound = confusion_matrix(val_labels, preds_compound)

record_result('Compound_Final_300px', f1_compound_final, 'Compound')
print(f"\nCompound F1 (@300px): {f1_compound_final:.4f}")
print(f"Per-class: [{f1_compound_per[0]:.4f}, {f1_compound_per[1]:.4f}, {f1_compound_per[2]:.4f}]")
print(f"\nConfusion Matrix:")
print(cm_compound)
print(f"\nNormalized:")
print((cm_compound.astype(float) / cm_compound.sum(axis=1, keepdims=True)).round(4))

del model_compound
torch.cuda.empty_cache()
gc.collect()

Experiment B: Compound (Progressive Resolution)

--- Stage 1: 128x128 ---


  E1: val_f1=0.9440 (best=0.9440)


  E2: val_f1=0.9468 (best=0.9468)


  E3: val_f1=0.9524 (best=0.9524)


  E4: val_f1=0.9540 (best=0.9540)


...

  E8: val_f1=0.9756 (best=0.9756)


  E9: val_f1=0.9764 (best=0.9764)


  E10: val_f1=0.9767 (best=0.9767)
  ✅ Compound_300px: 0.9767


  ✅ Compound_Final_300px: 0.9767

Compound F1 (@300px): 0.9767
Per-class: [0.9679, 0.9855, 0.9769]

Confusion Matrix:
[[1942   10   48]
 [   8  782    2]
 [  63    3 2448]]

Normalized:
[[0.971  0.005  0.024 ]
 [0.0101 0.9874 0.0025]
 [0.0251 0.0012 0.9737]]


0

---
## Comparison with Current Champion

In [13]:
print("=" * 60)
print("Final Summary")
print("=" * 60)

# Load yuda_8 champion for comparison
summary_yuda8 = pd.read_csv(RESULTS / 'yuda8_summary.csv')
champion_yuda8 = summary_yuda8.iloc[0]['best_val_f1']

print(f"\n{'Method':50s} {'F1':8s}")
print('-' * 60)
print(f"{'yuda_8 champion (Stacking All 6)':50s} {champion_yuda8:.4f}")

# Hierarchical
hier_r = [r for r in ALL_RESULTS if r['method'] == 'Hierarchical' and 'Pipeline' in r['name']]
if hier_r:
    print(f"{'Hierarchical (Electronic→R+O)':50s} {hier_r[0]['best_val_f1']:.4f}")

# Compound
comp_r = [r for r in ALL_RESULTS if r['method'] == 'Compound' and 'Final' in r['name']]
if comp_r:
    print(f"{'Compound (128→224→300px)':50s} {comp_r[0]['best_val_f1']:.4f}")

# Print all stages
print(f"\n{'=' * 60}")
print("All Results:")
df_all = pd.DataFrame(ALL_RESULTS).sort_values('best_val_f1', ascending=False).reset_index(drop=True)
print(df_all.to_string(index=False))

df_all.to_csv(RESULTS / 'yuda9_summary.csv', index=False)

Final Summary

Method                                             F1      
------------------------------------------------------------
yuda_8 champion (Stacking All 6)                   0.9872
Hierarchical (Electronic→R+O)                      0.9826
Compound (128→224→300px)                           0.9767

All Results:
                 name  best_val_f1       method
  Hierarchical_Stage1     0.996655 Hierarchical
Hierarchical_Pipeline     0.982636 Hierarchical
  Hierarchical_Stage2     0.979136 Hierarchical
 Compound_Final_300px     0.976739     Compound
       Compound_300px     0.976739     Compound
       Compound_224px     0.974699     Compound
       Compound_128px     0.960300     Compound
